In [5]:
import gzip
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from pymatgen.core import Element, Structure
import warnings
warnings.filterwarnings("ignore")  # For any minor warnings

# 1. Load data
with gzip.open("matbench_perovskites.json.gz", "rt") as f:
    raw = json.load(f)

df = pd.DataFrame(raw["data"], columns=raw["columns"])
structures = [Structure.from_dict(s) for s in df["structure"]]
targets = df["e_form"].values  # eV/cell (total) for Matbench compatibility
# If you want eV/atom: targets = targets / np.array([s.num_sites for s in structures])

print(f"Loaded {len(structures)} perovskites")
print(f"Target range (eV/cell): {targets.min():.3f} – {targets.max():.3f}")

Loaded 18928 perovskites
Target range (eV/cell): -0.640 – 5.160


In [6]:
# 2. Graph construction (same as before, but with 128 RBF bins)
def get_atomic_number(symbol: str) -> int:
    return Element(symbol).Z

def structure_to_graph(struct_dict, cutoff=8.0):
    lattice = np.array(struct_dict["lattice"]["matrix"])
    sites = struct_dict["sites"]
    n_atoms = len(sites)
    frac_coords = np.array([site["abc"] for site in sites])
    cart_coords = frac_coords @ lattice
    elements = [site["species"][0]["element"] for site in sites]
    atom_nums = [get_atomic_number(el) for el in elements]

    src, dst, dists = [], [], []
    for i in range(n_atoms):
        for j in range(n_atoms):
            if i == j: continue
            for nx in [-1, 0, 1]:
                for ny in [-1, 0, 1]:
                    for nz in [-1, 0, 1]:
                        shift = np.array([nx, ny, nz]) @ lattice
                        dvec = cart_coords[j] + shift - cart_coords[i]
                        dist = np.linalg.norm(dvec)
                        if 0 < dist <= cutoff:
                            src.append(i)
                            dst.append(j)
                            dists.append(dist)
    return atom_nums, src, dst, dists

def gaussian_rbf(dists, cutoff=8.0, n_bins=128):  # Increased bins for better features
    dists = np.asarray(dists, dtype=np.float32)
    centers = np.linspace(0.1, cutoff, n_bins)
    width = centers[1] - centers[0]
    x = dists[:, None] - centers[None, :]
    return np.exp(-(x**2) / (2 * width**2)).astype(np.float32)

# 3. Prepare data list (same)
data_list = []
for i, struct_dict in enumerate(df["structure"]):
    atom_nums, src, dst, dists = structure_to_graph(struct_dict, cutoff=8.0)
    if len(dists) == 0: continue
    edge_feat = gaussian_rbf(dists, cutoff=8.0, n_bins=128)
    data_list.append((
        torch.LongTensor(atom_nums),
        torch.LongTensor(src),
        torch.LongTensor(dst),
        torch.FloatTensor(edge_feat),
        torch.FloatTensor([targets[i]]),
        len(atom_nums)
    ))

print(f"Processed {len(data_list)} graphs")

Processed 18928 graphs


In [7]:
# 4. Updated Model with Dropout and BatchNorm
class SimpleMPNNLayer(nn.Module):
    def __init__(self, atom_dim, edge_dim, hidden_dim):
        super().__init__()
        self.message_net = nn.Sequential(
            nn.Linear(2 * atom_dim + edge_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.update_net = nn.Sequential(
            nn.Linear(atom_dim + hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
            nn.Dropout(0.1),
        )

    def forward(self, h, src, dst, edge_attr):
        msg = torch.cat([h[src], h[dst], edge_attr], dim=-1)
        msg = self.message_net(msg)
        num_nodes, hidden = h.size(0), msg.size(1)
        aggr = torch.zeros(num_nodes, hidden, device=h.device, dtype=h.dtype)
        aggr.scatter_add_(dim=0, index=dst.unsqueeze(1).expand(-1, hidden), src=msg)
        h_new = self.update_net(torch.cat([h, aggr], dim=-1))
        return h_new

class PerovskiteGNN(nn.Module):
    def __init__(self, atom_fea_dim=92, edge_fea_dim=128, hidden_dim=128, n_layers=4):  # Reduced hidden_dim
        super().__init__()
        self.atom_embed = nn.Embedding(100, atom_fea_dim)
        self.convs = nn.ModuleList([
            SimpleMPNNLayer(atom_fea_dim if i == 0 else hidden_dim, edge_fea_dim, hidden_dim)
            for i in range(n_layers)
        ])
        self.readout = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, atom_nums, src, dst, edge_attr, batch_num_nodes):
        h = self.atom_embed(atom_nums)
        for i, conv in enumerate(self.convs):
            h_new = conv(h, src, dst, edge_attr)
            if i > 0:
                h = h_new + h
            else:
                h = h_new
        graph_feats = []
        offset = 0
        for n in batch_num_nodes:
            graph_feats.append(h[offset:offset + n].mean(dim=0))
            offset += n
        graph_feats = torch.stack(graph_feats)
        out = self.readout(graph_feats).squeeze(-1)
        return out

# 5. Collate (same)
def collate_graphs(batch_indices):
    atom_all, src_all, dst_all, edge_all, y_all, n_all = [], [], [], [], [], []
    node_offset = 0
    for idx in batch_indices:
        a, s, d, e, y, n = data_list[idx]
        atom_all.append(a)
        src_all.append(s + node_offset)
        dst_all.append(d + node_offset)
        edge_all.append(e)
        y_all.append(y)
        n_all.append(n)
        node_offset += n
    return (
        torch.cat(atom_all),
        torch.cat(src_all),
        torch.cat(dst_all),
        torch.cat(edge_all),
        torch.cat(y_all),
        n_all
    )

In [9]:
# 6. Training with 5-fold CV
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
MAX_EPOCHS = 100  # Reduced
PATIENCE = 20  # For early stopping
kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold_maes = []
for fold, (train_val_idx, test_idx) in enumerate(kf.split(range(len(data_list)))):
    print(f"\nFold {fold+1}/5")
    train_idx, val_idx = train_test_split(train_val_idx, test_size=0.1111, random_state=42)
    
    model = PerovskiteGNN(edge_fea_dim=128, hidden_dim=128).to(device)
    optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)  # Increased WD
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    criterion = nn.MSELoss()
    
    best_val_mae = float('inf')
    patience_counter = 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_loss = 0.0
        np.random.shuffle(train_idx)
        for i in range(0, len(train_idx), BATCH_SIZE):
            batch_idx = train_idx[i:i + BATCH_SIZE]
            a, s, d, e, y, n_nodes = collate_graphs(batch_idx)
            a, s, d, e, y = a.to(device), s.to(device), d.to(device), e.to(device), y.to(device)
            pred = model(a, s, d, e, n_nodes)
            loss = criterion(pred, y)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            train_loss += loss.item() * len(batch_idx)
        train_loss /= len(train_idx)
        
        # Validation with metric check
        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad():
            for i in range(0, len(val_idx), BATCH_SIZE):
                batch_idx = val_idx[i:i + BATCH_SIZE]
                a, s, d, e, y, n_nodes = collate_graphs(batch_idx)
                a, s, d, e, y = a.to(device), s.to(device), d.to(device), e.to(device), y.to(device)
                pred = model(a, s, d, e, n_nodes)
                val_preds.extend(pred.cpu().tolist())
                val_true.extend(y.cpu().tolist())
        
        val_preds_np = np.array(val_preds)
        val_true_np = np.array(val_true)
        val_mae = mean_absolute_error(val_true_np, val_preds_np)
        val_rmse = mean_squared_error(val_true_np, val_preds_np)
        
        # Sanity check for metrics
        if val_mae > val_rmse:
            print(f"WARNING: MAE {val_mae:.4f} > RMSE {val_rmse:.4f} - possible bug! Recalculating...")
            errors = np.abs(val_true_np - val_preds_np)
            recal_mae = np.mean(errors)
            recal_rmse = np.sqrt(np.mean(errors**2))
            print(f"Recalculated: MAE {recal_mae:.4f}, RMSE {recal_rmse:.4f}")
            val_mae, val_rmse = recal_mae, recal_rmse  # Override if needed
        
        scheduler.step(val_mae)
        print(f"Epoch {epoch:3d} | train MSE {train_loss:.5f} | val MAE {val_mae:.4f} | val RMSE {val_rmse:.4f}")
        
        if val_mae < best_val_mae:
            best_val_mae = val_mae
            torch.save(model.state_dict(), f"best_model_fold{fold}.pt")
            patience_counter = 0
            print(f"  → new best val MAE: {best_val_mae:.4f}")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print("Early stopping triggered")
                break
    
    # Test on fold
    model.load_state_dict(torch.load(f"best_model_fold{fold}.pt"))
    model.eval()
    test_preds, test_true = [], []
    with torch.no_grad():
        for i in range(0, len(test_idx), BATCH_SIZE):
            batch_idx = test_idx[i:i + BATCH_SIZE]
            a, s, d, e, y, n_nodes = collate_graphs(batch_idx)
            a, s, d, e = a.to(device), s.to(device), d.to(device), e.to(device)
            pred = model(a, s, d, e, n_nodes)
            test_preds.extend(pred.cpu().tolist())
            test_true.extend(y.cpu().tolist())
    
    test_mae = mean_absolute_error(test_true, test_preds)
    fold_maes.append(test_mae)
    print(f"Fold {fold+1} test MAE: {test_mae:.4f}")


Fold 1/5
Recalculated: MAE 0.2975, RMSE 0.3982
Epoch   1 | train MSE 0.32971 | val MAE 0.2975 | val RMSE 0.3982
  → new best val MAE: 0.2975
Recalculated: MAE 0.2280, RMSE 0.2961
Epoch   2 | train MSE 0.14488 | val MAE 0.2280 | val RMSE 0.2961
  → new best val MAE: 0.2280
Recalculated: MAE 0.1958, RMSE 0.2519
Epoch   3 | train MSE 0.08617 | val MAE 0.1958 | val RMSE 0.2519
  → new best val MAE: 0.1958
Recalculated: MAE 0.1721, RMSE 0.2289
Epoch   4 | train MSE 0.06762 | val MAE 0.1721 | val RMSE 0.2289
  → new best val MAE: 0.1721
Recalculated: MAE 0.1505, RMSE 0.1978
Epoch   5 | train MSE 0.05730 | val MAE 0.1505 | val RMSE 0.1978
  → new best val MAE: 0.1505
Recalculated: MAE 0.1304, RMSE 0.1768
Epoch   6 | train MSE 0.04932 | val MAE 0.1304 | val RMSE 0.1768
  → new best val MAE: 0.1304
Recalculated: MAE 0.1268, RMSE 0.1649
Epoch   7 | train MSE 0.04772 | val MAE 0.1268 | val RMSE 0.1649
  → new best val MAE: 0.1268
Recalculated: MAE 0.1251, RMSE 0.1656
Epoch   8 | train MSE 0.0415

In [10]:
# Final results
mean_mae = np.mean(fold_maes)
std_mae = np.std(fold_maes)
print("\n" + "="*60)
print(f"Mean test MAE across 5 folds: {mean_mae:.4f} ± {std_mae:.4f} eV/cell")
print("="*60)


Mean test MAE across 5 folds: 0.0541 ± 0.0030 eV/cell
